In [2]:
# =========================================
# 🔹 0. Import Libraries
# =========================================
import pandas as pd
import numpy as np

# =========================================
# 🔹 1. Load Dataset
# =========================================
df = pd.read_excel("shopping_dataset.xlsx")

print("Shape:", df.shape)
print("Columns:", df.columns)

df.head()
df.info()

# 👉 Set your target column (CHANGE THIS)
target = df.columns[-1]   # or manually set: "Purchased", "Price", etc.

# =========================================
# 🔹 2. Handle Missing Values
# =========================================
print("\nMissing Values:\n", df.isnull().sum())

# Numerical → mean
for col in df.select_dtypes(include=np.number):
    df[col].fillna(df[col].mean(), inplace=True)

# Categorical → mode
for col in df.select_dtypes(include='object'):
    df[col].fillna(df[col].mode()[0], inplace=True)

# =========================================
# 🔹 3. Remove Duplicates
# =========================================
print("Duplicates:", df.duplicated().sum())
df.drop_duplicates(inplace=True)

# =========================================
# 🔹 4. Handle Outliers (SAFE CAPPING)
# =========================================
for col in df.select_dtypes(include=np.number):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])

# =========================================
# 🔹 5. Fix Data Types
# =========================================
print("\nBefore Types:\n", df.dtypes)

for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass

print("\nAfter Types:\n", df.dtypes)

# =========================================
# 🔹 6. Categorical Encoding (SMART)
# =========================================
from sklearn.preprocessing import LabelEncoder

for col in df.select_dtypes(include='object'):
    unique_vals = df[col].nunique()

    if unique_vals == 2:
        # Binary → Label Encoding
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])

    elif unique_vals < 10:
        # Nominal → One-Hot
        df = pd.get_dummies(df, columns=[col], drop_first=True)

    else:
        # Too many categories → drop (for your small dataset)
        df.drop(col, axis=1, inplace=True)

# =========================================
# 🔹 7. Feature Scaling
# =========================================
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = scaler.fit_transform(df[num_cols])

# =========================================
# 🔹 8. Remove Irrelevant / Redundant Features
# =========================================

# Remove constant columns
for col in df.columns:
    if df[col].nunique() <= 1:
        df.drop(col, axis=1, inplace=True)

# Remove highly correlated features
corr = df.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]
df.drop(columns=to_drop, inplace=True)

# =========================================
# 🔹 9. Handle Skewness
# =========================================
for col in df.select_dtypes(include=np.number):
    if abs(df[col].skew()) > 1:
        df[col] = np.log1p(df[col] - df[col].min() + 1)

# =========================================
# ✅ Final Output
# =========================================
print("\nFinal Shape:", df.shape)
df.head()
df.info()

Shape: (2000, 7)
Columns: Index(['PurchaseID', 'CustomerID', 'Item', 'Category', 'Price', 'Discount(%)',
       'FinalPrice'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PurchaseID   2000 non-null   int64  
 1   CustomerID   2000 non-null   object 
 2   Item         2000 non-null   object 
 3   Category     2000 non-null   object 
 4   Price        2000 non-null   int64  
 5   Discount(%)  2000 non-null   int64  
 6   FinalPrice   2000 non-null   float64
dtypes: float64(1), int64(3), object(3)
memory usage: 109.5+ KB

Missing Values:
 PurchaseID     0
CustomerID     0
Item           0
Category       0
Price          0
Discount(%)    0
FinalPrice     0
dtype: int64
Duplicates: 0

Before Types:
 PurchaseID     float64
CustomerID      object
Item            object
Category        object
Price          float64
Discount

/tmp/ipykernel_13023/2534894897.py:28: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mean(), inplace=True)
/tmp/ipykernel_13023/2534894897.py:32: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try 


Final Shape: (2000, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   PurchaseID            2000 non-null   float64
 1   Price                 2000 non-null   float64
 2   Discount(%)           2000 non-null   float64
 3   Category_Bags         2000 non-null   bool   
 4   Category_Beauty       2000 non-null   bool   
 5   Category_Clothing     2000 non-null   bool   
 6   Category_Electronics  2000 non-null   bool   
 7   Category_Footwear     2000 non-null   bool   
dtypes: bool(5), float64(3)
memory usage: 56.8 KB
